In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

root = Path.cwd()
while not (root / "data" / "raw").exists() and root != root.parent:
    root = root.parent
DATA = root / "data" / "raw"
airports = pd.read_csv(DATA / "AA_AIRPORT_INFO.csv")
pax = pd.read_csv(DATA / "AA_PAX_FORECASTING.csv")

TARGET = "OD_PAX"
YEAR_COL = "DEP_YEAR"
MONTH_COL = "DEP_MONTH"
DATE = "date"

In [ ]:
df = pax.dropna(subset=[TARGET]).copy()
df[DATE] = pd.to_datetime(dict(year=df[YEAR_COL], month=df[MONTH_COL], day=1))
df["year"] = df[YEAR_COL]
df["month"] = df[MONTH_COL]

df = df.merge(airports.add_prefix("ORG_"), left_on="ORG_AIRPRT", right_on="ORG_AIRPRT_CD", how="left")
df = df.merge(airports.add_prefix("DES_"), left_on="DES_AIRPRT", right_on="DES_AIRPRT_CD", how="left")

print(df.shape)

In [ ]:
prev = df[["ORG_AIRPRT", "DES_AIRPRT", DATE, TARGET]].copy()
prev[DATE] = prev[DATE] + pd.DateOffset(years=1)
prev = prev.rename(columns={TARGET: "pax_last_year"})
df = df.merge(prev, on=["ORG_AIRPRT", "DES_AIRPRT", DATE], how="left")

print("Rows with a last-year value:", df["pax_last_year"].notna().mean().round(3) * 100, "%")

In [ ]:
FEATURES = ["year", "month", "ORG_AIRPRT", "DES_AIRPRT", "pax_last_year"]

X = df[FEATURES].copy()
for c in X.select_dtypes("object"):
    X[c] = X[c].astype("category").cat.codes
X = X.fillna(-1)
y = df[TARGET]

CUTOFF = df[DATE].max() - pd.DateOffset(months=11)
train = df[DATE] < CUTOFF
test = ~train
print("Train rows:", train.sum(), " Test rows:", test.sum())
print("Train:", df.loc[train, DATE].min().date(), "to", df.loc[train, DATE].max().date())
print("Test:", df.loc[test, DATE].min().date(), "to", df.loc[test, DATE].max().date())

In [ ]:
def score(name, actual, pred):
    print(f"{name:20s} MAE: {mean_absolute_error(actual, pred):,.1f}   "
          f"RMSE: {mean_squared_error(actual, pred) ** 0.5:,.1f}")

score("Last-year baseline", y[test], df.loc[test, "pax_last_year"].fillna(0))

rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                           max_samples=0.3, n_jobs=-1, random_state=42)
rf.fit(X[train], y[train])
score("Random Forest", y[test], rf.predict(X[test]))

In [ ]:
print(pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3))

In [ ]:
## Random Forest — Candidate Model (Lukas)

### Validation and metrics
- Time-based split: train on data before last 12 months, test on the last 12 months. No future data is used in training.
- Metrics: MAE and RMSE (shared team metrics)

### Results
| Model | MAE | RMSE |
| Last-year baseline | MAE: 547.4 |  RMSE: 1,366.0 |
| Random Forest | MAE: 523.9 |  RMSE: 1,259.1 |
## Trade-offs and limitations
- Only slightly more accurate than the simple baseline, but much more complex and slower.
- Can't predict higher than the highest value it saw in training.

### Conclusion
We can carry Random Forest forward as the baseline model: it beats the naive forecast, is reproducible, and can improve with more features. Any other model used must beat both RF and the last-year baseline to justify its added complexity.